# SQL Query Modernization with Mellea

This notebook demonstrates how to modernize legacy SQL queries using Mellea's Instruct-Validate-Repair pattern.

## Scenario

We have a legacy SQL query for tennis statistics with a hardcoded player name (Roger Federer). We'll:
1. Create an in-memory SQLite database with test data
2. Execute the original query with hardcoded values
3. Use Mellea to modernize the query (add new features, parameterize)
4. Validate that the modernized query works correctly

## Setup

In [4]:
# Install required packages if needed
# !pip install mellea sqlparse

In [6]:
import sqlite3
from typing import Literal
import sqlparse
import pydantic
from mellea import generative, start_session
from mellea.backends import ModelOption
from mellea.stdlib.requirement import req, simple_validate
from mellea.stdlib.sampling import RepairTemplateStrategy
import pandas as pd

## Step 1: Create In-Memory Database with Test Data

We'll create two tables:
- `player_stats`: Basic player statistics
- `match_details`: Additional match details (for modernized query)

In [7]:
# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create player_stats table
cursor.execute('''
CREATE TABLE player_stats (
    player_id INTEGER PRIMARY KEY,
    player_name TEXT NOT NULL,
    year INTEGER,
    total_matches INTEGER,
    wins INTEGER,
    losses INTEGER,
    win_percentage REAL
)
''')

# Create match_details table
cursor.execute('''
CREATE TABLE match_details (
    detail_id INTEGER PRIMARY KEY,
    player_id INTEGER,
    fastest_ace_speed REAL,
    most_points_won_against TEXT,
    most_points_lost_against TEXT,
    FOREIGN KEY (player_id) REFERENCES player_stats(player_id)
)
''')

print("✓ Tables created successfully")

✓ Tables created successfully


## Step 2: Insert Roger Federer Test Data

In [8]:
# Insert Roger Federer's statistics for multiple years
federer_stats = [
    (1, 'Roger Federer', 2023, 45, 38, 7, 84.4),
    (2, 'Roger Federer', 2022, 52, 42, 10, 80.8),
    (3, 'Roger Federer', 2021, 38, 30, 8, 78.9),
]

cursor.executemany('''
    INSERT INTO player_stats (player_id, player_name, year, total_matches, wins, losses, win_percentage)
    VALUES (?, ?, ?, ?, ?, ?, ?)
''', federer_stats)

# Insert match details for Roger Federer
federer_details = [
    (1, 1, 135.2, 'Rafael Nadal', 'Novak Djokovic'),
    (2, 2, 138.5, 'Andy Murray', 'Rafael Nadal'),
    (3, 3, 132.8, 'Novak Djokovic', 'Dominic Thiem'),
]

cursor.executemany('''
    INSERT INTO match_details (detail_id, player_id, fastest_ace_speed, most_points_won_against, most_points_lost_against)
    VALUES (?, ?, ?, ?, ?)
''', federer_details)

conn.commit()
print("✓ Test data inserted successfully")

# Display the data
print("\nPlayer Stats:")
df_stats = pd.read_sql_query("SELECT * FROM player_stats", conn)
display(df_stats)

print("\nMatch Details:")
df_details = pd.read_sql_query("SELECT * FROM match_details", conn)
display(df_details)

✓ Test data inserted successfully

Player Stats:


,player_id,player_name,year,total_matches,wins,losses,win_percentage
0,1,Roger Federer,2023,45,38,7,84.4
1,2,Roger Federer,2022,52,42,10,80.8
2,3,Roger Federer,2021,38,30,8,78.9



Match Details:


,detail_id,player_id,fastest_ace_speed,most_points_won_against,most_points_lost_against
0,1,1,135.2,Rafael Nadal,Novak Djokovic
1,2,2,138.5,Andy Murray,Rafael Nadal
2,3,3,132.8,Novak Djokovic,Dominic Thiem


## Step 3: Define the Original (Legacy) Query

This query has a hardcoded player name and limited columns.

In [9]:
ORIGINAL_QUERY = """
SELECT 
    player_name,
    total_matches,
    wins,
    losses,
    win_percentage
FROM player_stats
WHERE player_name = 'Roger Federer'
ORDER BY year DESC;
"""

print("Original Query:")
print(ORIGINAL_QUERY)

Original Query:

SELECT 
    player_name,
    total_matches,
    wins,
    losses,
    win_percentage
FROM player_stats
WHERE player_name = 'Roger Federer'
ORDER BY year DESC;



## Step 4: Execute Original Query

In [10]:
# Execute the original query
df_original = pd.read_sql_query(ORIGINAL_QUERY, conn)
print("Results from Original Query:")
display(df_original)

Results from Original Query:


,player_name,total_matches,wins,losses,win_percentage
0,Roger Federer,45,38,7,84.4
1,Roger Federer,52,42,10,80.8
2,Roger Federer,38,30,8,78.9


## Step 5: Define Validation Functions

These functions ensure the modernized query meets our requirements.

**Key validation approaches:**
- **SQL Syntax**: Uses SQLite's `EXPLAIN QUERY PLAN` to validate actual SQL semantics (not just parsing)
- **Backward Compatibility**: Simple string matching for required columns
- **Hardcoded Parameters**: Uses a `@generative` function that returns structured data about detected hardcoded values

In [11]:
# Expected columns from original query (for backward compatibility)
REQUIRED_COLUMNS = {"player_name", "total_matches", "wins", "losses", "win_percentage"}

# Define response model for hardcoded parameter detection
class HardcodedParameterCheck(pydantic.BaseModel):
    contains_hardcoded: Literal["yes", "no"]
    hardcoded_value: str | None = None  # The actual hardcoded value found, if any

def validate_sql_syntax(sql: str) -> bool:
    """Validate SQL syntax by attempting to prepare the query with SQLite."""
    try:
        # First check if it can be parsed
        parsed = sqlparse.parse(sql)
        if not (parsed and parsed[0].tokens):
            return False
        
        # Create a temporary in-memory connection for validation (thread-safe)
        import sqlite3
        temp_conn = sqlite3.connect(':memory:', check_same_thread=False)
        temp_cursor = temp_conn.cursor()
        
        # Create the same table structure for validation
        temp_cursor.execute('''
            CREATE TABLE player_stats (
                player_id INTEGER PRIMARY KEY,
                player_name TEXT,
                year INTEGER,
                total_matches INTEGER,
                wins INTEGER,
                losses INTEGER,
                win_percentage REAL
            )
        ''')
        temp_cursor.execute('''
            CREATE TABLE match_details (
                detail_id INTEGER PRIMARY KEY,
                player_id INTEGER,
                fastest_ace_speed REAL,
                most_points_won_against TEXT,
                most_points_lost_against TEXT
            )
        ''')
        
        # Replace parameter placeholders for validation
        test_sql = sql.replace(':player_name', "'test'")
        test_sql = test_sql.replace('?', "'test'")
        
        # Use EXPLAIN to validate without executing
        temp_cursor.execute(f"EXPLAIN QUERY PLAN {test_sql}")
        temp_conn.close()
        return True
    except Exception as e:
        print(f"SQL validation failed: {e}")
        return False

def validate_backward_compatible(sql: str) -> bool:
    """Ensure new query includes all original columns."""
    sql_upper = sql.upper()
    return all(col.upper() in sql_upper for col in REQUIRED_COLUMNS)

# Define the generative function for checking hardcoded parameters
@generative
def contains_hardcoded_parameters(sql_query: str) -> HardcodedParameterCheck:
    """Determines if the SQL query contains hardcoded values for parameters such as player names, player ids, dates etc. Instead of specific names, id or dates, the SQL query must use variables such as player_name, player_id, date etc.
    
    Returns:
        HardcodedParameterCheck with:
        - contains_hardcoded: 'yes' if hardcoded parameters found, 'no' otherwise
        - hardcoded_value: The actual hardcoded value found (e.g., 'Roger Federer'), or None if no hardcoded values
    """

print("✓ Validation functions defined")

✓ Validation functions defined


## Step 6: Check if Modernization is Needed

In [12]:
@generative
def needs_modernization_or_contains_hardcoded_parameter(sql_query: str) -> Literal["yes", "no"]:
    """Determines if the SQL query is outdated or has hardcoded parameter. If it has hardcoded parameter it needs modernization. Parameters can be things like player names, player ids, dates etc."""

# Start Mellea session
m = start_session()

# Check if query needs modernization
needs_update = needs_modernization_or_contains_hardcoded_parameter(m, sql_query=ORIGINAL_QUERY)
print(f"Needs modernization? {needs_update}")

=== 13:20:24-WARNING ======
Calling the function with NO strategy BUT requirements. No requirement is being checked!
Needs modernization? yes


## Step 7: Modernize the Query with Mellea

We'll use Mellea's Instruct-Validate-Repair pattern to:
1. Add new columns from the `match_details` table
2. Replace hardcoded values with parameters
3. Ensure backward compatibility

In [13]:
# Create a validation wrapper that uses the generative function
def validate_no_hardcoded_params_wrapper(sql: str) -> bool:
    """Wrapper that uses generative validation to check for hardcoded parameters."""
    result = contains_hardcoded_parameters(m, sql_query=sql)
    
    # Display what was found
    if result.contains_hardcoded == "yes":
        print(f"⚠️  Hardcoded parameter detected: {result.hardcoded_value}")
    else:
        print("✓ No hardcoded parameters found")
    
    return result.contains_hardcoded == "no"  # Return True if NO hardcoded params found

if needs_update == "yes":
    # Modernize the query with validation
    result = m.instruct(
        """Modernize this SQL query to add new features:
        - fastest_ace_speed (from match_details table)
        - most_points_won_against (opponent name)
        - most_points_lost_against (opponent name)
        
        Join with match_details table on player_id.
        Replace hardcoded player name with :player_name parameter.
        
        Return ONLY the SQL query.""",
        grounding_context={"original_query": ORIGINAL_QUERY},
        requirements=[
            req(
                "SQL must have valid syntax",
                validation_fn=simple_validate(validate_sql_syntax)
            ),
            req(
                "SQL must include all original columns",
                validation_fn=simple_validate(validate_backward_compatible)
            ),
            req(
                "SQL must use :player_name parameter instead of hardcoded values (validated using @generative)",
                validation_fn=simple_validate(validate_no_hardcoded_params_wrapper)
            ),
        ],
        strategy=RepairTemplateStrategy(loop_budget=10),
        return_sampling_results=True,
    )
    
    print("\nModernized Query:")
    print(result.result)
    
    if result.success:
        print(f"\n✓ Successfully modernized after {len(result.sample_generations)} attempt(s)")
        modernized_query = str(result.result)
    else:
        print(f"\n✗ Failed after {len(result.sample_generations)} attempts")
        modernized_query = None
else:
    print("Query is already modern!")
    modernized_query = None

  0%|          | 0/10 [00:00<?, ?it/s]

SQL validation failed: near "Here": syntax error
=== 13:20:36-WARNING ======
Calling the function with NO strategy BUT requirements. No requirement is being checked!
⚠️  Hardcoded parameter detected: player_name
=== 13:20:37-INFO ======
FAILED. Valid: 1/3


 10%|█         | 1/10 [00:06<00:54,  6.05s/it]

=== 13:20:39-WARNING ======
Calling the function with NO strategy BUT requirements. No requirement is being checked!
✓ No hardcoded parameters found
=== 13:20:40-INFO ======
SUCCESS


 10%|█         | 1/10 [00:08<01:18,  8.68s/it]


Modernized Query:
SELECT 
    player_name,
    total_matches,
    wins,
    losses,
    win_percentage,
    fastest_ace_speed,
    most_points_won_against,
    most_points_lost_against
FROM player_stats ps
JOIN match_details md ON ps.player_id = md.player_id
WHERE player_name = :player_name
ORDER BY year DESC;

✓ Successfully modernized after 2 attempt(s)


## Step 8: Execute Modernized Query with Parameters

Now we'll execute the modernized query using a parameter for the player name.

In [14]:
if modernized_query:
    # SQLite uses ? for parameters, so we need to replace :player_name with ?
    sqlite_query = modernized_query.replace(':player_name', '?')
    
    print("Executing modernized query with parameter: 'Roger Federer'")
    print(f"\nQuery to execute:\n{sqlite_query}")
    
    try:
        # Execute with parameter
        df_modernized = pd.read_sql_query(sqlite_query, conn, params=('Roger Federer',))
        print("\nResults from Modernized Query:")
        display(df_modernized)
    except Exception as e:
        print(f"\n✗ Error executing modernized query: {e}")
        print("\nAttempting to fix query...")
        # If there's an error, we might need to adjust the query
        df_modernized = None
else:
    print("No modernized query to execute")
    df_modernized = None

Executing modernized query with parameter: 'Roger Federer'

Query to execute:
SELECT 
    player_name,
    total_matches,
    wins,
    losses,
    win_percentage,
    fastest_ace_speed,
    most_points_won_against,
    most_points_lost_against
FROM player_stats ps
JOIN match_details md ON ps.player_id = md.player_id
WHERE player_name = ?
ORDER BY year DESC;

Results from Modernized Query:


,player_name,total_matches,wins,losses,win_percentage,fastest_ace_speed,most_points_won_against,most_points_lost_against
0,Roger Federer,45,38,7,84.4,135.2,Rafael Nadal,Novak Djokovic
1,Roger Federer,52,42,10,80.8,138.5,Andy Murray,Rafael Nadal
2,Roger Federer,38,30,8,78.9,132.8,Novak Djokovic,Dominic Thiem


## Step 9: Validate Results

Let's verify that the modernized query:
1. Returns the same core data as the original query
2. Includes the new columns we requested

In [15]:
if df_modernized is not None:
    print("Validation Results:")
    print("=" * 50)
    
    # Check 1: All original columns present
    original_cols = set(df_original.columns)
    modernized_cols = set(df_modernized.columns)
    
    if original_cols.issubset(modernized_cols):
        print("✓ All original columns present in modernized query")
    else:
        missing = original_cols - modernized_cols
        print(f"✗ Missing columns: {missing}")
    
    # Check 2: New columns added
    expected_new_cols = {'fastest_ace_speed', 'most_points_won_against', 'most_points_lost_against'}
    new_cols_present = expected_new_cols.intersection(modernized_cols)
    
    if new_cols_present:
        print(f"✓ New columns added: {new_cols_present}")
    else:
        print("✗ No new columns added")
    
    # Check 3: Same number of rows
    if len(df_original) == len(df_modernized):
        print(f"✓ Same number of rows: {len(df_original)}")
    else:
        print(f"✗ Row count mismatch: {len(df_original)} vs {len(df_modernized)}")
    
    # Check 4: Core data matches
    print("\nComparing core statistics:")
    for col in ['player_name', 'total_matches', 'wins', 'losses', 'win_percentage']:
        if col in df_modernized.columns:
            if df_original[col].equals(df_modernized[col]):
                print(f"  ✓ {col} matches")
            else:
                print(f"  ✗ {col} differs")
    
    print("\n" + "=" * 50)
    print("\nSummary:")
    print(f"Original query columns: {list(df_original.columns)}")
    print(f"Modernized query columns: {list(df_modernized.columns)}")
else:
    print("Cannot validate - modernized query did not execute successfully")

Validation Results:
✓ All original columns present in modernized query
✓ New columns added: {'most_points_won_against', 'fastest_ace_speed', 'most_points_lost_against'}
✓ Same number of rows: 3

Comparing core statistics:
  ✓ player_name matches
  ✓ total_matches matches
  ✓ wins matches
  ✓ losses matches
  ✓ win_percentage matches


Summary:
Original query columns: ['player_name', 'total_matches', 'wins', 'losses', 'win_percentage']
Modernized query columns: ['player_name', 'total_matches', 'wins', 'losses', 'win_percentage', 'fastest_ace_speed', 'most_points_won_against', 'most_points_lost_against']


## Step 10: Test with Different Parameters

Let's verify the parameterization works by testing with a different player (if we had data).

In [16]:
if modernized_query:
    print("Testing parameterization:")
    print("\nThe modernized query can now be used with any player name:")
    print(f"  cursor.execute(query, {{'player_name': 'Roger Federer'}})")
    print(f"  cursor.execute(query, {{'player_name': 'Rafael Nadal'}})")
    print(f"  cursor.execute(query, {{'player_name': 'Novak Djokovic'}})")
    print("\n✓ Query is now reusable and maintainable!")

Testing parameterization:

The modernized query can now be used with any player name:
  cursor.execute(query, {'player_name': 'Roger Federer'})
  cursor.execute(query, {'player_name': 'Rafael Nadal'})
  cursor.execute(query, {'player_name': 'Novak Djokovic'})

✓ Query is now reusable and maintainable!


## Cleanup

In [17]:
# Close database connection
conn.close()
print("✓ Database connection closed")

✓ Database connection closed


## Summary

This notebook demonstrated:

1. **Creating test data**: Set up an in-memory SQLite database with Roger Federer's statistics
2. **Original query**: Executed a legacy query with hardcoded values
3. **Mellea modernization**: Used the Instruct-Validate-Repair pattern to:
   - Add new columns from joined tables
   - Replace hardcoded values with parameters
   - Ensure backward compatibility
4. **Validation**: Verified the modernized query returns correct results

### Key Benefits of the IVR Pattern:

- **Syntax validation**: Ensures generated SQL is valid
- **Backward compatibility**: Preserves all original columns
- **Parameterization**: Makes queries reusable and secure (validated using `@generative` function)
- **Automatic repair**: Retries up to 3 times if validation fails
- **Generative validation**: Uses LLM-based checks for complex requirements like detecting hardcoded values

### Next Steps:

- Add more test data for other players
- Extend validation to check query performance
- Add more complex modernization requirements
- Test with different database systems